In [1]:
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter
from os import listdir
from os.path import (
    join as join_path,
    isdir
)

from json import load as load_json

In [2]:
def load_json_dict(path):
    try :
        with open(path) as f:
            d = load_json(f)
    except FileNotFoundError:
        return None
    return d
def update_results_dict(loss_fn_name, lam_name, dname, expno, results, eval_tracker):
    if not loss_fn_name in results:
        results[loss_fn_name] = {}
    if not lam_name in results[loss_fn_name]:
        results[loss_fn_name][lam_name] = {}
    if not dname in results[loss_fn_name][lam_name]:
        results[loss_fn_name][lam_name][dname] = {}
    if not expno in results[loss_fn_name][lam_name][dname]:
        results[loss_fn_name][lam_name][dname][expno] = {}
    results[loss_fn_name][lam_name][dname][expno] = eval_tracker
    return results

import numpy as np

def summarize_results(huge_dict):
    summary = {}

    for loss_name, labeling_methods in huge_dict.items():
        for labeling_method, datasets in labeling_methods.items():
            for dataset_name, experiments in datasets.items():
                if dataset_name not in summary:
                    summary[dataset_name] = {}

                if loss_name not in summary[dataset_name]:
                    summary[dataset_name][loss_name] = {}

                if labeling_method not in summary[dataset_name][loss_name]:
                    summary[dataset_name][loss_name][labeling_method] = {}

                metrics_values = {}
                labeled_pts = []
                dataset_sizes = []
                cluster_counts = []
                iteration_counts = []

                for exp_num, evals in experiments.items():
                    if evals is None:
                        continue
                    for metric, values in evals.items():
                        if values is None:
                            continue

                        if metric == "no_labeled_pts":
                            if isinstance(values, list) and values:
                                labeled_pts.append(values[-1])
                                iteration_counts.append(len(values))  # Number of iterations
                        elif metric == "dataset_size":
                            if isinstance(values, list) and values:
                                dataset_sizes.append(values[-1])
                            elif isinstance(values, (int, float)):
                                dataset_sizes.append(values)
                        elif metric == "predicted_labels":
                            if isinstance(values, list) and values:
                                unique_labels = np.unique(values)
                                clusters = unique_labels[unique_labels != -1]
                                cluster_counts.append(len(clusters))
                        else:
                            if isinstance(values, list) and values:
                                metrics_values.setdefault(metric, []).append(values[-1])

                # Compute and store labeled ratio and dataset size
                if labeled_pts and dataset_sizes:
                    avg_labeled = np.mean(labeled_pts)
                    std_labeled = np.std(labeled_pts)
                    avg_dataset_size = np.mean(dataset_sizes)
                    labeled_ratio_str = f"{avg_labeled:.2f}±{std_labeled:.2f} /{int(avg_dataset_size)} = {100 * avg_labeled / avg_dataset_size:.2f}%"
                    summary[dataset_name][loss_name][labeling_method]["labeled_ratio"] = labeled_ratio_str
                    summary[dataset_name][loss_name][labeling_method]["dataset_size"] = int(avg_dataset_size)

                # Number of clusters
                if cluster_counts:
                    mean_clusters = np.mean(cluster_counts)
                    std_clusters = np.std(cluster_counts)
                    summary[dataset_name][loss_name][labeling_method]["num_clusters"] = f"{mean_clusters:.2f}±{std_clusters:.2f}"

                # Number of iterations
                if iteration_counts:
                    mean_iter = np.mean(iteration_counts)
                    std_iter = np.std(iteration_counts)
                    summary[dataset_name][loss_name][labeling_method]["num_iterations"] = f"{mean_iter:.2f}±{std_iter:.2f}"

                # Compute other metrics
                for metric, values in metrics_values.items():
                    mean = np.mean(values)
                    std = np.std(values)
                    summary_str = f"{mean:.2f}±{std:.2f}"
                    summary[dataset_name][loss_name][labeling_method][metric] = summary_str

                # Handle case where no metrics are available
                if not metrics_values and experiments:
                    summary[dataset_name][loss_name][labeling_method] = "-"

    return summary



In [3]:
experiment_path_1 = "/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison404"
loss_funs = listdir(experiment_path_1)
for lf in loss_funs:
    _path = join_path(experiment_path_1, lf)
    if not isdir(_path):
        loss_funs.remove(lf)

In [4]:
results_dict = {}
for loss_fn_name in loss_funs:
    label_assignment_methods_names = listdir(join_path(
        experiment_path_1, loss_fn_name
    ))
    for lam_name in label_assignment_methods_names:
        datasets_names = listdir(join_path(
            experiment_path_1, loss_fn_name, lam_name
        ))
        for dname in datasets_names:
            experiments_numbers = listdir(join_path(
                experiment_path_1, loss_fn_name, lam_name, dname
            ))
            for expno in experiments_numbers:
                trackers_path = join_path(
                    experiment_path_1, loss_fn_name, lam_name, dname, expno, "trackers"
                )
                evalt_path = join_path(trackers_path, "eval_tracker.json")
                eval_tracker = load_json_dict(evalt_path)
                results_dict = update_results_dict(loss_fn_name, lam_name, dname, expno, results_dict, eval_tracker)

In [5]:
results_dict["ae_sync_loss"]["knn_label_assignment"]["example"]["model_00"].keys()

dict_keys(['ari_labeled', 'ari_total', 'ami_labeled', 'ami_total', 'predicted_labels', 'no_labeled_pts', 'dataset_size'])

### if you have another results folder

In [6]:
experiment_path_2 = "/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison405"
loss_funs = listdir(experiment_path_2)
for lf in loss_funs:
    _path = join_path(experiment_path_2, lf)
    if not isdir(_path):
        loss_funs.remove(lf)

In [7]:
for loss_fn_name in loss_funs:
    label_assignment_methods_names = listdir(join_path(
        experiment_path_2, loss_fn_name
    ))
    for lam_name in label_assignment_methods_names:
        datasets_names = listdir(join_path(
            experiment_path_2, loss_fn_name, lam_name
        ))
        for dname in datasets_names:
            experiments_numbers = listdir(join_path(
                experiment_path_2, loss_fn_name, lam_name, dname
            ))
            for expno in experiments_numbers:
                trackers_path = join_path(
                    experiment_path_2, loss_fn_name, lam_name, dname, expno, "trackers"
                )
                evalt_path = join_path(trackers_path, "eval_tracker.json")
                eval_tracker = load_json_dict(evalt_path)
                results_dict = update_results_dict(loss_fn_name, lam_name, dname, expno, results_dict, eval_tracker)

In [8]:
results_dict["ae_sync_loss"]["knn_label_assignment"]["example"]["model_00"]["dataset_size"]

8000

### Summary

In [9]:
summary = summarize_results(results_dict)

In [10]:
summary["example"]["ae_sync_loss"]["knn_label_assignment"]

{'labeled_ratio': '7655.80±99.66 /8000 = 95.70%',
 'dataset_size': 8000,
 'num_clusters': '4.00±0.00',
 'num_iterations': '72.40±12.26',
 'ari_labeled': '1.00±0.00',
 'ari_total': '0.94±0.02',
 'ami_labeled': '0.99±0.00',
 'ami_total': '0.93±0.02'}

In [11]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font

def save_summary_as_excel_tables(summary_dict, output_file='summary_tables.xlsx'):
    # Prepare table structures
    metrics = ['ari_total', 'ari_labeled', 'ami_total', 'ami_labeled', 'labeled_ratio', 'num_clusters', 'num_iterations']
    tables = {metric: {} for metric in metrics}
    datasets = summary_dict.keys()

    # Build the tables
    for dataset in datasets:
        losses = summary_dict[dataset]
        for loss, methods in losses.items():
            for method, scores in methods.items():
                row_name = f"{loss}+{method}"
                for metric in metrics:
                    if row_name not in tables[metric]:
                        tables[metric][row_name] = {}
                    tables[metric][row_name][dataset] = scores.get(metric, None)

    # Write all tables to a single sheet
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        workbook = writer.book
        sheet_name = 'Results Summary'
        writer.sheets[sheet_name] = workbook.create_sheet(title=sheet_name)
        sheet = writer.sheets[sheet_name]
        
        row_offset = 0

        for metric in metrics:
            # Add a title row
            title = metric.replace('_', ' ').upper() + " RESULTS"
            sheet.cell(row=row_offset + 1, column=1, value=title).font = Font(bold=True, size=14)

            # Convert table to DataFrame
            df = pd.DataFrame.from_dict(tables[metric], orient='index')
            df.index.name = 'Method'
            df.reset_index(inplace=True)

            # Write DataFrame to sheet
            for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), start=row_offset + 2):
                for c_idx, value in enumerate(row, start=1):
                    sheet.cell(row=r_idx, column=c_idx, value=value)

            # Update row offset (table height + space)
            row_offset += len(df) + 4

        # Remove default sheet if it exists
        if 'Sheet' in workbook.sheetnames:
            std = workbook['Sheet']
            workbook.remove(std)

        workbook.save(output_file)
        print(f"Results saved to {output_file}")


In [12]:
save_summary_as_excel_tables(summary, output_file=join_path(experiment_path_1, "results_summary_comparison404and405 - k.xlsx"))

Results saved to /export/share/peters57dm/Verbund/deepsync/results/experiments/comparison404/results_summary_comparison404and405 - k.xlsx


# Old saving method.

In [14]:
import pandas as pd


def parse_mean_std(value):
    try:
        mean_str, std_str = value.split("±")
        return float(mean_str), float(std_str)
    except:
        return None, None

def save_highlighted_summary(summary_dict, save_path):
    all_rows = []
    metrics_set = set()

    # Prepare raw data from summary_dict
    for dataset, losses in summary_dict.items():
        for loss_name, label_methods in losses.items():
            for label_method, metrics in label_methods.items():
                row = {
                    "Dataset": dataset,
                    "Loss": loss_name,
                    "Labeling Method": label_method
                }
                if isinstance(metrics, dict):
                    for metric, value in metrics.items():
                        row[metric] = value
                        metrics_set.add(metric)
                else:
                    row["Result"] = "-"
                all_rows.append(row)

    columns = ["Dataset", "Loss", "Labeling Method"] + sorted(metrics_set)
    df = pd.DataFrame(all_rows, columns=columns).fillna("-")
    df.to_excel(save_path, index=False)

    # Load workbook and sheet
    wb = load_workbook(save_path)
    ws = wb.active

    # Styling
    header_font = Font(bold=True)
    ami_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")  # Light green
    ari_fill = PatternFill(start_color="FFFACD", end_color="FFFACD", fill_type="solid")  # Light yellow
    orange_fill = PatternFill(start_color="FFD580", end_color="FFD580", fill_type="solid")  # Light orange
    gray_fill = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")  # Gray

    # Bold headers and auto-width
    for col in ws.columns:
        max_length = max(len(str(cell.value)) if cell.value else 0 for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max_length + 2
        ws.cell(row=1, column=col[0].column).font = header_font

    # Column indexes
    ami_col = columns.index("ami_total") + 1 if "ami_total" in columns else None
    ari_col = columns.index("ari_total") + 1 if "ari_total" in columns else None
    num_cols = len(columns)

    df["RowNum"] = df.index + 2  # Excel row numbers
    recommendations = {}

    # Shift rows after inserting blank lines
    offset = 0
    for dataset in df["Dataset"].unique():
        dataset_rows = df[df["Dataset"] == dataset]
        row_indices = list(dataset_rows.index + 2 + offset)

        for metric_name, metric_col, fill in [
            ("ami_total", ami_col, ami_fill),
            ("ari_total", ari_col, ari_fill)
        ]:
            if metric_col:
                scored_rows = []
                for idx in dataset_rows.index:
                    row = df.loc[idx]
                    value = row.get(metric_name, "-")
                    mean, std = parse_mean_std(value)
                    if mean is not None:
                        row_num = row["RowNum"] + offset
                        scored_rows.append((mean, std, row_num, row["Loss"], row["Labeling Method"]))

                if scored_rows:
                    sorted_rows = sorted(scored_rows, key=lambda x: (-x[0], x[1]))

                    # Best
                    if len(sorted_rows) >= 1:
                        mean, std, row_num, loss, label = sorted_rows[0]
                        ws.cell(row=int(row_num), column=metric_col).fill = fill
                        recommendations.setdefault(dataset, []).append((metric_name, loss, label))

                    # Second-best
                    if len(sorted_rows) >= 2:
                        _, _, row_num2, _, _ = sorted_rows[1]
                        ws.cell(row=int(row_num2), column=metric_col).fill = orange_fill

        # Add empty gray row after dataset
        insert_row = max(row_indices) + 1
        ws.insert_rows(insert_row)

        for col in range(1, num_cols + 1):
            cell = ws.cell(row=insert_row, column=col)
            cell.fill = gray_fill
            cell.value = ""

        offset += 1  # Track row shift due to insertion

    wb.save(save_path)
    print(f"\n✅ Excel with highlights and section spacers saved to: {save_path}")

    # === Per Dataset Recommendation ===
    print("\n=== Recommendations per Dataset ===")
    for dataset, bests in recommendations.items():
        count = {}
        for metric, loss, label in bests:
            key = (loss, label)
            count[key] = count.get(key, 0) + 1

        # Most frequent winner
        best_comb = max(count.items(), key=lambda x: x[1])[0]
        print(f"Dataset: {dataset} ➜ Recommended: Loss = '{best_comb[0]}', Labeling Method = '{best_comb[1]}' (based on {count[best_comb]} metric(s))")

    # === Final Overall Recommendation ===
    overall_metric_winners = {}

    for dataset, bests in recommendations.items():
        for metric, loss, label in bests:
            key = (loss, label)
            if metric not in overall_metric_winners:
                overall_metric_winners[metric] = {}
            overall_metric_winners[metric][key] = overall_metric_winners[metric].get(key, 0) + 1

    print("\n=== Final Overall Recommendations ===")
    for metric, combo_counts in overall_metric_winners.items():
        best_combo = max(combo_counts.items(), key=lambda x: x[1])
        loss_name, label_method = best_combo[0]
        wins = best_combo[1]
        print(f"{metric}: Loss = '{loss_name}', Labeling Method = '{label_method}' (Won in {wins} dataset(s))")

In [14]:
save_highlighted_summary(summary, join_path(experiment_path, "results_summary_comparison301.xlsx"))


✅ Excel with highlights and section spacers saved to: /export/share/peters57dm/Verbund/deepsync/experiments/comparison301/results_summary_comparison301.xlsx

=== Recommendations per Dataset ===
Dataset: easy_blobs ➜ Recommended: Loss = 'ae_sync_loss', Labeling Method = 'mahalanobis_label_assignment' (based on 2 metric(s))
Dataset: example ➜ Recommended: Loss = 'ae_sync_loss', Labeling Method = 'knn_average_dist_label_assignment' (based on 2 metric(s))
Dataset: USPS ➜ Recommended: Loss = 'ae_sync_loss', Labeling Method = 'knn_average_dist_label_assignment' (based on 2 metric(s))
Dataset: htru ➜ Recommended: Loss = 'ae_sync_loss', Labeling Method = 'knn_label_assignment' (based on 2 metric(s))
Dataset: pendigits ➜ Recommended: Loss = 'att_rep_loss', Labeling Method = 'knn_average_dist_label_assignment' (based on 2 metric(s))
Dataset: optdigits ➜ Recommended: Loss = 'ae_sync_loss', Labeling Method = 'mahalanobis_label_assignment' (based on 2 metric(s))
Dataset: letterrecognition ➜ Recomm